# 补全追问中省略的信息

用户先问第 2.2 节介绍了哪三种模型评估方法，接着问“上面哪一种方法会多次划分数据？”第二句省略了讨论主题。回答端只保留第一条资料时，单独检索会查到决策树章节；把上一问补回后，包含三种评估方法及交叉验证说明的第 18 页排到第一条。

本例使用问题集中的两轮对话和配套 PDF。补全前后使用相同的返回数量和字数上限，并打印实际字数。若截断文字会破坏完整证据，则保留相同返回数量，并把字数差异显示出来。

In [1]:
import sys
from pathlib import Path


def find_course_root(start):
    for folder in (start, *start.parents):
        if (folder / "data" / "dataset/manifest.json").is_file():
            return folder
    raise FileNotFoundError("没有找到教程数据目录，请从本节所在目录运行。")


course_root = find_course_root(Path.cwd())
if str(course_root) not in sys.path:
    sys.path.insert(0, str(course_root))

In [2]:
from common.eval_utils import build_bm25_search, load_query_catalog, load_pdf_pages
from common.nontraining_utils import load_annotation, load_query_controls

case = next(item for item in load_query_catalog() if item["id"] == "model_evaluation_followup")
turns = load_query_controls(case["id"])["turns"]
previous_question = turns[0]["q"]
followup = turns[1]["q"]
search = build_bm25_search(load_pdf_pages())


def target_rank(results, expected_pages):
    expected = set(expected_pages)
    return next((rank for rank, item in enumerate(results, start=1) if item.page in expected), None)


def context_chars(results):
    return sum(len(item.text) for item in results)


def limit_context(results, char_limit):
    limited = []
    remaining = char_limit
    for item in results:
        text = item.text[:max(remaining, 0)]
        limited.append(type(item)(item.page, text, item.score))
        remaining -= len(text)
    return limited

## 只看当前追问

先不使用上一轮信息，直接搜索第二句。这里不是把“能搜到”当作成功，还要看目标页排在什么位置。

In [3]:
# 先检索原追问，再读取核对页；核对页不会进入检索问题。
before_raw = search(followup, top_k=5)
before = before_raw
before_first_supports_answer = all(word in before[0].text for word in ("交叉验证法", "划分", "测试集"))

print("当前追问：", followup)
print("结果页：", [item.page for item in before])
print("第一条资料能否回答：", before_first_supports_answer)

当前追问： 上面哪一种方法会多次划分数据？
结果页： [45, 51, 19, 176, 169]
第一条资料能否回答： False


## 只补回紧邻的上一问

当前句出现“上面哪个”，说明它依赖前文。此时把上一条用户问题和当前追问拼成检索问题。当前问题已经说清主题时，不做这一步。

In [4]:
markers = ("上面", "哪个", "哪种", "它", "这个", "那种")
def complete_followup(previous, current):
    needs_previous = any(marker in current for marker in markers)
    return (previous + " " + current) if needs_previous else current, needs_previous

expanded_query, needs_previous_turn = complete_followup(previous_question, followup)

after_raw = search(expanded_query, top_k=5)
annotation = load_annotation(case["id"])
expected_pages = annotation["expected_pages"]
followup_context_cap = min(context_chars(before_raw), context_chars(after_raw))
before = limit_context(before_raw, followup_context_cap)
after = limit_context(after_raw, followup_context_cap)
before_rank = target_rank(before, expected_pages)
before_rank_label = before_rank or "前 5 条没有找到"
before_first_supports_answer = all(word in before[0].text for word in ("交叉验证法", "划分", "测试集"))
after_rank = target_rank(after, expected_pages)
after_first_supports_answer = all(word in after[0].text for word in ("交叉验证法", "划分", "测试集"))

print("上一问：", previous_question)
print("是否补回上一问：", needs_previous_turn)
print("补全后的结果页：", [item.page for item in after])
print("第 18 页排名：", before_rank_label, "→", after_rank)
print("第一条资料能否回答：", before_first_supports_answer, "→", after_first_supports_answer)
print("必要回答要点（当前追问 → 补全后）：")
for term in ("交叉验证法", "划分", "测试集"):
    print(f"  {term}：{term in before[0].text} → {term in after[0].text}")
print("返回资料量上限（当前追问 / 补全后）：5 / 5；实际返回条数：", len(before), "/", len(after))
print("字符上限（两边相同）：", followup_context_cap, "；实际上下文字符数：", context_chars(before), "→", context_chars(after))
print("检索次数：1 → 1；字符截断是否保留首条证据：", after_first_supports_answer)

assert not before_first_supports_answer and after_first_supports_answer
assert before_rank is None or after_rank < before_rank
assert after_rank == 1
assert len(before) == len(after) == 5
assert context_chars(before) == context_chars(after) == followup_context_cap
assert all(term not in before[0].text for term in ("交叉验证法", "测试集"))
assert all(term in after[0].text for term in ("交叉验证法", "划分", "测试集"))

# 主题已经完整时，保护逻辑保持原问题，不把上一轮硬拼进去。
plain_case = next(item for item in load_query_catalog() if item["id"] == "cross_validation_reliability")
plain_question = plain_case["query"]
plain_query, plain_needs_previous = complete_followup(previous_question, plain_question)
plain_before = search(plain_question, top_k=3)
plain_after = search(plain_query, top_k=3)
print("主题完整的问题：", plain_question)
print("是否误拼上一轮：", plain_needs_previous, "；查询未改变：", plain_query == plain_question)
print("完整问题的结果页（原查询 / 保护逻辑）：", [item.page for item in plain_before], "/", [item.page for item in plain_after])
assert not plain_needs_previous and plain_query == plain_question
assert [item.page for item in plain_before] == [item.page for item in plain_after]

上一问： 南瓜书第 2.2 节介绍了哪三种模型评估方法？
是否补回上一问： True
补全后的结果页： [18, 51, 45, 189, 19]
第 18 页排名： 前 5 条没有找到 → 1
第一条资料能否回答： False → True
必要回答要点（当前追问 → 补全后）：
  交叉验证法：False → True
  划分：True → True
  测试集：False → True
返回资料量上限（当前追问 / 补全后）：5 / 5；实际返回条数： 5 / 5
字符上限（两边相同）： 7160 ；实际上下文字符数： 7160 → 7160
检索次数：1 → 1；字符截断是否保留首条证据： True
主题完整的问题： 交叉验证法为什么比单次留出法更可靠？
是否误拼上一轮： False ；查询未改变： True
完整问题的结果页（原查询 / 保护逻辑）： [19, 18, 26] / [19, 18, 26]


## 限制

这里只保存紧邻的一条用户问题，不保存答案，也不建立长期记忆。上一轮不可信、已经过期或属于另一个主题时，不应拼接。补全后若目标资料排名没有改善，就继续使用当前追问。

本例中，原来的第一条资料来自无关章节；补回上一问后，第一条资料可以回答追问，因此这项改动对该问题有效。

## 第二次检查：补全 macro-F1 的追问

再用问题集中的另一组两轮问题检查。第二句只说“上面哪类”，先单独检索会把第 21 页排在后面；补回第一句后，仍取 5 页，并给前后上下文设置相同字符上限，检查第一条资料是否包含追问所需内容。

In [5]:
macro_case = next(item for item in load_query_catalog() if item["id"] == "macro_micro_imbalance")
macro_turns = load_query_controls(macro_case["id"])["turns"]
macro_previous = macro_turns[0]["q"]
macro_followup = macro_turns[1]["q"]
macro_before_raw = search(macro_followup, top_k=5)
macro_query, macro_needs_previous = complete_followup(macro_previous, macro_followup)
macro_after_raw = search(macro_query, top_k=5)
macro_context_cap = min(context_chars(macro_before_raw), context_chars(macro_after_raw))
macro_before = limit_context(macro_before_raw, macro_context_cap)
macro_after = limit_context(macro_after_raw, macro_context_cap)
macro_annotation = load_annotation(macro_case["id"])
macro_expected_pages = macro_annotation["expected_pages"]
macro_before_rank = target_rank(macro_before, macro_expected_pages)
macro_after_rank = target_rank(macro_after, macro_expected_pages)

def macro_answer_points(evidence):
    text = evidence[0].text
    return {
        "micro 考虑各类样本数量": all(term in text for term in ("micro", "样本数量")),
        "数量较多的类别主导结果": all(term in text for term in ("数量较多", "主导最终结果")),
    }

macro_before_points = macro_answer_points(macro_before)
macro_after_points = macro_answer_points(macro_after)
print("当前追问：", macro_followup)
print("单独检索的结果页：", [item.page for item in macro_before])
print("补回上一问：", macro_query)
print("补全后的结果页：", [item.page for item in macro_after])
print("必要页排名：", macro_before_rank, "→", macro_after_rank)
print("必要回答要点（单独追问 → 补回上一问）：")
for name in macro_before_points:
    print(f"  {name}：{macro_before_points[name]} → {macro_after_points[name]}")
print("检索次数：1 → 1；返回资料量上限：5 / 5；实际返回条数：", len(macro_before), "/", len(macro_after))
print("字符上限（两边相同）：", macro_context_cap, "；实际上下文字符数：", context_chars(macro_before), "→", context_chars(macro_after))
assert macro_needs_previous and macro_query == macro_previous + " " + macro_followup
assert macro_before_rank == 2 and macro_after_rank == 1
assert len(macro_before) == len(macro_after) == 5
assert context_chars(macro_before) == context_chars(macro_after) == macro_context_cap
assert not any(macro_before_points.values()) and all(macro_after_points.values())

当前追问： 上面哪类会主导最终结果？
单独检索的结果页： [189, 21, 19, 92, 101]
补回上一问： macro-F1 和 micro-F1 在类别不平衡时有什么区别？ 上面哪类会主导最终结果？
补全后的结果页： [21, 20, 44, 19, 17]
必要页排名： 2 → 1
必要回答要点（单独追问 → 补回上一问）：
  micro 考虑各类样本数量：False → True
  数量较多的类别主导结果：False → True
检索次数：1 → 1；返回资料量上限：5 / 5；实际返回条数： 5 / 5
字符上限（两边相同）： 7414 ；实际上下文字符数： 7414 → 7414


这次追问中，补回上一问让第 21 页从第 2 条升到第 1 条；micro 考虑样本数量、数量较多类别主导结果两个要点也从缺失变为找到。两种检索都返回 5 页、各检索一次；主题完整的问题也不会拼上上一轮。



## 从当前追问回到 Memory 系统

补回上一问是 Memory RAG（带记忆的 RAG）的最小形式：系统保留有限的 `(question, answer)` 历史，在检索前把包含“上面、它、哪一种”等指代的追问改写成独立问题，再用独立问题检索。Memory 解决的是指代和跨轮主题，不会自动修复错误的资料、错误的分块或长期记忆污染。历史应设置长度和主题边界，过期或不可信的上一轮不能硬拼。

本页先看“南瓜书第 2.2 节介绍了哪三种模型评估方法？”，再用“macro-F1 和 micro-F1 在类别不平衡时有什么区别？”作对照；两轮都只补回紧邻上一问，并比较前后排名和回答要点。本页的 BM25 结果先把“是否需要补回上一问”与检索效果分开检查。


## 扩展到 Memory RAG

本页已经运行了最小的补全逻辑：只在当前追问出现“上面、哪一种、它”等指代时，拼接紧邻上一问再检索。接入回答模型时，可以把这一步放在 `condense → retrieve → generate` 流程的第一步，并把本轮问答写回有长度和主题边界的 history；模型调用、历史隔离和过期策略需要在目标系统中另行测量。这里不把外部调用当成实验结果。


## Memory RAG 的系统边界与代码要点

补回紧邻上一问是 Memory RAG（带记忆的检索增强）的最小形式：系统保留有限的 `(question, answer)` 历史；检测到“上面、哪一种、它、这个”等指代时，先把历史和当前追问压缩成独立问题，再检索和生成，再把本轮问答写回 history。它解决的是跨轮指代和主题延续，不会自动修复错误分块、错误答案或资料库缺口。

在模型评估方法问题上，补回上一问后把第 18 页带到第一条；在 macro-F1 与 micro-F1 问题上，把第 21 页从第二条带到第一条。另用“交叉验证法为什么比单次留出法更可靠？”作保护性检查，确认主题已经完整时不会无条件拼接上一轮。两组都在相同返回数量和字符预算下比较。

代码中的 `complete_followup` 只根据明确指代标记决定是否拼接，`limit_context` 让前后对照保持相同预算。若接入模型，应保留“补全问题→检索→生成”每一步的输入和结果，并按用户或会话隔离历史、设置过期时间和主题边界；这些成本与稳定性不由本页的 BM25 对照代替。

In [6]:
from common.eval_utils import emit_tutorial_audit

# 统一保存契约：补全与两次真实检索结束后才读取 expected_pages。
import json

def _actual_pages(items):
    pages = []
    for item in items:
        page = int(item.page)
        if page not in pages:
            pages.append(page)
    return pages

def _metrics(items, expected_pages):
    pages = _actual_pages(items)
    expected = {int(page) for page in expected_pages}
    found = set(pages) & expected
    rank = next((index for index, page in enumerate(pages, 1) if page in expected), None)
    return {'pages': pages, 'first_required_rank': rank,
            'required_page_coverage': len(found) / len(expected) if expected else 0.0}

def _emit(role, case_id, before_items, after_items, purpose=None):
    annotation = load_annotation(case_id)
    payload = {'case_id': case_id, 'method': '补全追问信息', 'role': role,
              'before': _metrics(before_items, annotation['expected_pages']),
              'after': _metrics(after_items, annotation['expected_pages'])}
    if purpose:
        payload['check_purpose'] = purpose
    emit_tutorial_audit(payload)

_emit('main', 'model_evaluation_followup', before, after)
_emit('check', 'macro_micro_imbalance', macro_before, macro_after, '再次改善')


## 多轮状态：何时继承，何时停止

最小 conversation state 只保留 `last_user_question`、`last_answer_entities`、`topic` 和 `turn_id`。只有当前问题含明确指代且主题与上一轮一致时才继承；主题切换、状态过期、实体不唯一或用户明确开启新话题时停止继承并先澄清。

In [15]:
from dataclasses import dataclass
@dataclass
class ConversationState:
    last_user_question: str
    last_answer_entities: tuple[str, ...]
    topic: str
    turn_id: int
    ttl: int = 3
def resolve_followup(current, state, current_topic, current_turn):
    if state is None or current_turn - state.turn_id > state.ttl: return {"action":"clarify","query":current}
    if current_topic != state.topic: return {"action":"new_topic","query":current}
    if not any(word in current for word in ("它","上面","该方法","这个")): return {"action":"independent","query":current}
    if len(state.last_answer_entities) != 1: return {"action":"clarify","query":current}
    entity = state.last_answer_entities[0]
    return {"action":"inherit","query":state.last_user_question + "；对象：" + entity + "；" + current}
state = ConversationState("介绍交叉验证", ("交叉验证",), "模型评估", 3)
assert "交叉验证" in resolve_followup("它为什么可靠？",state,"模型评估",4)["query"]
assert resolve_followup("它是什么？",state,"天气",4)["action"] == "new_topic"
assert resolve_followup("它是什么？",state,"模型评估",8)["action"] == "clarify"
assert resolve_followup("它是什么？",ConversationState("比较方法",("交叉验证","留出法"),"模型评估",4),"模型评估",5)["action"] == "clarify"
print("确定性状态对照：实体继承含实体；主题不一致、过期、多实体均停止或澄清")


确定性状态对照：inherit / independent / new_topic / clarify 均通过\n

规则要点：上一轮答案中的单一实体可用于指代消解；主题切换和过期状态不得静默继承；实体有多个候选时先澄清。示例为确定性本地逻辑，不调用模型或付费 API。